# Fine-tuning de Modelos Transformer para Deteccion de Cambio de Autor

## 0. Setup e Imports

In [1]:
# Importo todas las librerias necesarias para el fine-tuning
import torch
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from datasets import Dataset
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Configuro el estilo de los plots para que se vean bien
plt.style.use('seaborn-v0_8')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.precision', 4)

In [4]:
# Verifico que CUDA y la GPU esten disponibles
# Esto es critico porque el fine-tuning requiere GPU

print("="*70)
print("VERIFICACION DE HARDWARE")
print("="*70)

# Verificar CUDA
cuda_available = torch.cuda.is_available()
print(f"\nCUDA disponible: {cuda_available}")

if cuda_available:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    print(f"PyTorch version: {torch.__version__}")
    
    # Verificar VRAM libre
    torch.cuda.empty_cache()
    free_memory = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
    print(f"VRAM libre inicial: {free_memory / 1e9:.1f} GB")
else:
    print("\nADVERTENCIA: No se detecto GPU. El entrenamiento sera muy lento.")
    print("   Asegurate de tener CUDA y PyTorch con soporte GPU instalados.")

print("="*70)

VERIFICACION DE HARDWARE

CUDA disponible: True
GPU: NVIDIA GeForce RTX 5070 Ti
VRAM total: 17.1 GB
CUDA version: 12.8
PyTorch version: 2.11.0.dev20251227+cu128
VRAM libre inicial: 17.1 GB


In [5]:
# Defino las rutas del proyecto
# Necesito encontrar la raiz del proyecto para acceder a los datos y guardar checkpoints

def find_root() -> Path:
    """Busco la raiz del proyecto localizando el directorio data/raw."""
    current = Path.cwd()
    for cand in [current, *current.parents]:
        if (cand / "data" / "raw").exists():
            return cand
    raise FileNotFoundError("No se encontro data/raw desde el directorio actual.")

# Establezco todas las rutas que voy a necesitar
PROJECT_ROOT = find_root()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
BOUNDARIES_DIR = DATA_PROCESSED / "boundaries"
REPORTS_DIR = PROJECT_ROOT / "reports"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints" / "finetuning"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"BOUNDARIES_DIR: {BOUNDARIES_DIR}")
print(f"CHECKPOINTS_DIR: {CHECKPOINTS_DIR}")
print(f"REPORTS_DIR: {REPORTS_DIR}")

PROJECT_ROOT: /home/eeguskiza/DEUSTO/multi-author-analysis
BOUNDARIES_DIR: /home/eeguskiza/DEUSTO/multi-author-analysis/data/processed/boundaries
CHECKPOINTS_DIR: /home/eeguskiza/DEUSTO/multi-author-analysis/checkpoints/finetuning
REPORTS_DIR: /home/eeguskiza/DEUSTO/multi-author-analysis/reports


## 1. Configuracion Global

In [ ]:
# Configuracion global del experimento
# Aqui defino todos los parametros importantes para el fine-tuning

SEED = 42
MAX_LENGTH = 128  # Tokens maximos por par de oraciones

# Configuracion especifica por modelo
# Cada modelo tiene sus propios hyperparametros optimizados
CONFIGS = {
    "distilbert": {
        "model_name": "distilbert-base-uncased",
        "batch_size": 32,
        "learning_rate": 2e-5,
        "epochs": 3,
        "use_qlora": False,
        "description": "66M params - Full fine-tuning"
    },
    "qwen3-0.6b": {
        "model_name": "Qwen/Qwen3-0.6B",
        "batch_size": 16,
        "learning_rate": 2e-5,
        "epochs": 3,
        "use_qlora": False,
        "description": "600M params - Full fine-tuning"
    },
    "qwen3-1.7b": {
        "model_name": "Qwen/Qwen3-1.7B",
        "batch_size": 8,
        "learning_rate": 1e-4,  # Mayor LR para LoRA
        "epochs": 3,
        "use_qlora": True,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "description": "1.7B params - QLoRA (4-bit)"
    },
}

# Para entrenar solo algunos modelos, comentar los que no quieras
MODELS_TO_TRAIN = ["distilbert", "qwen3-0.6b", "qwen3-1.7b"]

# Muestro la configuracion
print("="*70)
print("CONFIGURACION DEL EXPERIMENTO")
print("="*70)
print(f"\nSEED: {SEED}")
print(f"MAX_LENGTH: {MAX_LENGTH} tokens")
print(f"\nModelos a entrenar: {len(MODELS_TO_TRAIN)}")
for model_key in MODELS_TO_TRAIN:
    config = CONFIGS[model_key]
    print(f"\n  {model_key}:")
    print(f"    {config['description']}")
    print(f"    Batch size: {config['batch_size']}")
    print(f"    Learning rate: {config['learning_rate']}")
    print(f"    Epochs: {config['epochs']}")
print("="*70)

CONFIGURACION DEL EXPERIMENTO

SEED: 42
MAX_LENGTH: 128 tokens

Modelos a entrenar: 4

  distilbert:
    66M params - Full fine-tuning
    Batch size: 32
    Learning rate: 2e-05
    Epochs: 3

  qwen3-0.6b:
    600M params - Full fine-tuning
    Batch size: 16
    Learning rate: 2e-05
    Epochs: 3

  qwen3-1.7b:
    1.7B params - QLoRA (4-bit)
    Batch size: 8
    Learning rate: 0.0001
    Epochs: 3

  qwen3-4b:
    4B params - QLoRA (4-bit)
    Batch size: 4
    Learning rate: 0.0001
    Epochs: 2


## 2. Carga de Datos y Creacion del Dataset

In [7]:
# Funcion para cargar oraciones desde los archivos raw
# Cada documento tiene una oracion por linea en los archivos .txt

def load_sentences(level: str, split: str, doc_id: str) -> list[str]:
    """Carga las oraciones de un documento desde el archivo raw.
    
    Args:
        level: 'easy', 'medium', o 'hard'
        split: 'train' o 'validation'
        doc_id: ID del documento (ej: 'problem-1')
        
    Returns:
        list[str]: Lista de oraciones del documento
    """
    path = DATA_RAW / level / split / f"{doc_id}.txt"
    
    if not path.exists():
        raise FileNotFoundError(f"No se encontro el archivo: {path}")
    
    # Leer y dividir por lineas (cada linea es una oracion)
    text = path.read_text(encoding='utf-8').strip()
    sentences = text.split('\n')
    
    # Filtrar lineas vacias
    sentences = [s.strip() for s in sentences if s.strip()]
    
    return sentences

print("Funcion load_sentences() definida.")

Funcion load_sentences() definida.


In [8]:
# Cargo los datasets de boundaries (pares de oraciones consecutivas con etiquetas)
# Cada fila contiene level, doc_id, sent_left_id, sent_right_id, y (0=mismo autor, 1=cambio)

print("Cargando boundaries...")
boundaries_train = pd.read_csv(BOUNDARIES_DIR / "boundaries_train.csv")
boundaries_val = pd.read_csv(BOUNDARIES_DIR / "boundaries_validation.csv")

print(f"\nTrain: {len(boundaries_train):,} boundaries")
print(f"Val: {len(boundaries_val):,} boundaries")

print(f"\nDistribucion train:")
print(boundaries_train['y'].value_counts())
print(f"  Clase 0 (mismo autor): {(boundaries_train['y'] == 0).mean():.1%}")
print(f"  Clase 1 (cambio):      {(boundaries_train['y'] == 1).mean():.1%}")

print(f"\nDistribucion validation:")
print(boundaries_val['y'].value_counts())
print(f"  Clase 0 (mismo autor): {(boundaries_val['y'] == 0).mean():.1%}")
print(f"  Clase 1 (cambio):      {(boundaries_val['y'] == 1).mean():.1%}")

print(f"\nDistribucion por nivel en validation:")
print(boundaries_val['level'].value_counts())

Cargando boundaries...

Train: 159,002 boundaries
Val: 33,858 boundaries

Distribucion train:
y
0    128708
1     30294
Name: count, dtype: int64
  Clase 0 (mismo autor): 80.9%
  Clase 1 (cambio):      19.1%

Distribucion validation:
y
0    27290
1     6568
Name: count, dtype: int64
  Clase 0 (mismo autor): 80.6%
  Clase 1 (cambio):      19.4%

Distribucion por nivel en validation:
level
medium    12863
hard      10749
easy      10246
Name: count, dtype: int64


In [9]:
# Funcion para crear pares de texto (sent_a, sent_b, label) desde el DataFrame de boundaries
# Esto es necesario para crear el Dataset de HuggingFace

def create_text_pairs(df: pd.DataFrame, desc: str = "Creando pares") -> list[dict]:
    """Crea lista de {text_a, text_b, label, level, doc_id} para el dataset.
    
    Args:
        df: DataFrame con boundaries (level, split, doc_id, sent_left_id, sent_right_id, y)
        desc: Descripcion para tqdm
        
    Returns:
        list[dict]: Lista de diccionarios con pares de oraciones y labels
    """
    pairs = []
    skipped = 0
    
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        try:
            # Cargar oraciones del documento
            sentences = load_sentences(row['level'], row['split'], row['doc_id'])
            
            # Validar que los indices existan
            if row['sent_left_id'] >= len(sentences) or row['sent_right_id'] >= len(sentences):
                skipped += 1
                continue
            
            # Obtener las dos oraciones
            text_a = sentences[row['sent_left_id']]
            text_b = sentences[row['sent_right_id']]
            
            # Validar que no esten vacias
            if not text_a or not text_b:
                skipped += 1
                continue
            
            # Anadir el par al dataset
            pairs.append({
                'text_a': text_a,
                'text_b': text_b,
                'label': int(row['y']),
                'level': row['level'],
                'doc_id': row['doc_id']
            })
            
        except Exception as e:
            skipped += 1
            if skipped <= 5:  # Mostrar solo los primeros errores
                print(f"\nError en {row['doc_id']}: {e}")
    
    if skipped > 0:
        print(f"\nSe saltaron {skipped} ejemplos por errores o indices invalidos")
    
    return pairs

print("Funcion create_text_pairs() definida.")

Funcion create_text_pairs() definida.


In [10]:
# Creo los datasets de HuggingFace para entrenamiento y validacion
# Este proceso tarda 5-10 minutos porque tiene que leer ~193k archivos

print("="*70)
print("CREANDO DATASETS")
print("="*70)
print("\nEste proceso puede tardar 5-10 minutos...\n")

# Crear pares de texto para entrenamiento
train_pairs = create_text_pairs(boundaries_train, desc="Creando pares de entrenamiento")
print(f"\n  Pares de entrenamiento validos: {len(train_pairs):,}")

# Crear pares de texto para validacion
val_pairs = create_text_pairs(boundaries_val, desc="Creando pares de validacion")
print(f"\n  Pares de validacion validos: {len(val_pairs):,}")

# Convertir a HuggingFace Dataset
print("\nConvirtiendo a HuggingFace Dataset...")
train_dataset = Dataset.from_list(train_pairs)
val_dataset = Dataset.from_list(val_pairs)

print("\nDatasets creados exitosamente")
print(f"\nTrain dataset: {len(train_dataset):,} ejemplos")
print(f"Val dataset: {len(val_dataset):,} ejemplos")
print("\nColumnas:", train_dataset.column_names)
print("\nEjemplo:")
print(train_dataset[0])
print("="*70)

CREANDO DATASETS

Este proceso puede tardar 5-10 minutos...



Creando pares de entrenamiento:   0%|          | 0/159002 [00:00<?, ?it/s]


Se saltaron 2307 ejemplos por errores o indices invalidos

  Pares de entrenamiento validos: 156,695


Creando pares de validacion:   0%|          | 0/33858 [00:00<?, ?it/s]


Se saltaron 560 ejemplos por errores o indices invalidos

  Pares de validacion validos: 33,298

Convirtiendo a HuggingFace Dataset...

Datasets creados exitosamente

Train dataset: 156,695 ejemplos
Val dataset: 33,298 ejemplos

Columnas: ['text_a', 'text_b', 'label', 'level', 'doc_id']

Ejemplo:
{'text_a': 'There\'s also incidents of "testosterone insensitive males" that have either ambiguous genitals or are phenotypically female.', 'text_b': 'Female is the human "default" body plan so a number of conditions exist that cause female-looking males.', 'label': 0, 'level': 'easy', 'doc_id': 'problem-1'}


## 3. Funciones de Metricas

In [11]:
# Funciones para calcular metricas de clasificacion
# Estas funciones seran usadas por el Trainer de HuggingFace

def compute_metrics(eval_pred):
    """Funcion de metricas para HuggingFace Trainer.
    
    Args:
        eval_pred: Tupla (logits, labels) del Trainer
        
    Returns:
        dict: Diccionario con metricas calculadas
    """
    predictions, labels = eval_pred
    
    # Obtener predicciones (clase con mayor probabilidad)
    predictions = np.argmax(predictions, axis=1)
    
    # Calcular metricas de clasificacion
    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)
    f1_class1 = f1_score(labels, predictions, average='binary', zero_division=0)  # F1 de clase 1 (cambio)
    precision_macro = precision_score(labels, predictions, average='macro', zero_division=0)
    recall_macro = recall_score(labels, predictions, average='macro', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_class1': f1_class1,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
    }

print("Funcion compute_metrics() definida.")

Funcion compute_metrics() definida.


## 4. Funciones de Tokenizacion

In [12]:
# Funciones de tokenizacion adaptadas al tipo de modelo
# Los modelos encoder (BERT) y decoder (Qwen) necesitan formatos diferentes

def get_tokenize_function(tokenizer, max_length: int = MAX_LENGTH):
    """Devuelve funcion de tokenizacion para modelos encoder (BERT-like).
    
    Args:
        tokenizer: Tokenizer de HuggingFace
        max_length: Longitud maxima en tokens
        
    Returns:
        function: Funcion de tokenizacion para dataset.map()
    """
    def tokenize_function(examples):
        # Para modelos encoder (BERT, DistilBERT, etc.)
        # Formato: [CLS] text_a [SEP] text_b [SEP]
        return tokenizer(
            examples['text_a'],
            examples['text_b'],
            truncation=True,
            max_length=max_length,
            padding=False,  # DataCollator se encarga del padding
        )
    return tokenize_function


def get_tokenize_function_causal(tokenizer, max_length: int = MAX_LENGTH):
    """Tokenizacion para modelos causales (Qwen, Llama, etc.).
    
    Concatena las oraciones con separadores explicitos.
    
    Args:
        tokenizer: Tokenizer de HuggingFace
        max_length: Longitud maxima en tokens
        
    Returns:
        function: Funcion de tokenizacion para dataset.map()
    """
    def tokenize_function(examples):
        # Para modelos causales, concatenar con formato explicito
        # Formato: "Sentence A: {text_a}\nSentence B: {text_b}"
        texts = [
            f"Sentence A: {a}\nSentence B: {b}"
            for a, b in zip(examples['text_a'], examples['text_b'])
        ]
        return tokenizer(
            texts,
            truncation=True,
            max_length=max_length,
            padding=False,
        )
    return tokenize_function

print("Funciones de tokenizacion definidas.")

Funciones de tokenizacion definidas.


## 5. Funcion de Entrenamiento Generica

In [13]:
# Funcion generica para entrenar cualquier modelo de la configuracion
# Soporta tanto full fine-tuning como QLoRA (4-bit) automaticamente

def train_model(
    model_key: str,
    train_dataset: Dataset,
    val_dataset: Dataset,
    output_dir: Path,
) -> dict:
    """
    Entrena un modelo y devuelve metricas.
    
    Args:
        model_key: Clave en CONFIGS (e.g., "distilbert", "qwen3-1.7b")
        train_dataset: Dataset de entrenamiento
        val_dataset: Dataset de validacion
        output_dir: Directorio para guardar checkpoints
    
    Returns:
        dict con metricas y configuracion del modelo
    """
    # Obtener configuracion del modelo
    config = CONFIGS[model_key]
    model_name = config["model_name"]
    use_qlora = config.get("use_qlora", False)
    
    print(f"\n{'='*70}")
    print(f"ENTRENANDO: {model_key}")
    print(f"{'='*70}")
    print(f"  Modelo base: {model_name}")
    print(f"  {config['description']}")
    print(f"  QLoRA: {use_qlora}")
    print(f"  Batch size: {config['batch_size']}")
    print(f"  Learning rate: {config['learning_rate']}")
    print(f"  Epochs: {config['epochs']}")
    
    # Limpiar VRAM antes de empezar
    torch.cuda.empty_cache()
    
    # 1. Cargar tokenizer
    print("\n[1/6] Cargando tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # 2. Determinar si es modelo causal (decoder) o encoder
    is_causal = "qwen" in model_name.lower() or "llama" in model_name.lower() or "gpt" in model_name.lower()
    print(f"   Tipo de modelo: {'Causal (Decoder)' if is_causal else 'Encoder'}")
    
    # 3. Cargar modelo
    print("\n[2/6] Cargando modelo...")
    if use_qlora:
        # Configuracion 4-bit para QLoRA
        # Esto reduce la VRAM necesaria de ~16GB a ~5-6GB para modelos grandes
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=2,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
        
        # Preparar para k-bit training
        model = prepare_model_for_kbit_training(model)
        
        # Configurar LoRA
        # LoRA entrena solo adaptadores pequenos en lugar de todos los pesos
        # Determinar target_modules segun el modelo
        if "qwen" in model_name.lower():
            target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]  # Qwen2
        else:
            target_modules = ["q_proj", "v_proj"]  # Default para Llama-like
        
        lora_config = LoraConfig(
            r=config.get("lora_r", 16),
            lora_alpha=config.get("lora_alpha", 32),
            lora_dropout=config.get("lora_dropout", 0.05),
            bias="none",
            task_type=TaskType.SEQ_CLS,
            target_modules=target_modules,
        )
        
        model = get_peft_model(model, lora_config)
        print("\n   Parametros entrenables con LoRA:")
        model.print_trainable_parameters()
        
    else:
        # Full fine-tuning - entrenar todos los parametros
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=2,
            trust_remote_code=True,
        )
        model.to("cuda")
        print(f"   Parametros totales: {sum(p.numel() for p in model.parameters()):,}")
        print(f"   Parametros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    
    # Configurar pad_token_id si no esta
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id
    
    # 4. Tokenizar datasets
    print("\n[3/6] Tokenizando datasets...")
    tokenize_fn = get_tokenize_function_causal(tokenizer) if is_causal else get_tokenize_function(tokenizer)
    
    train_tokenized = train_dataset.map(
        tokenize_fn, 
        batched=True, 
        remove_columns=['text_a', 'text_b'],  # Mantengo level y doc_id para analisis posterior
        desc="Tokenizando train"
    )
    val_tokenized = val_dataset.map(
        tokenize_fn, 
        batched=True, 
        remove_columns=['text_a', 'text_b'],
        desc="Tokenizando val"
    )
    
    print(f"   Train tokenizado: {len(train_tokenized):,} ejemplos")
    print(f"   Val tokenizado: {len(val_tokenized):,} ejemplos")
    
    # 5. Data collator para padding dinamico
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    # 6. Training arguments
    print("\n[4/6] Configurando entrenamiento...")
    training_args = TrainingArguments(
        output_dir=str(output_dir / model_key),
        num_train_epochs=config['epochs'],
        per_device_train_batch_size=config['batch_size'],
        per_device_eval_batch_size=config['batch_size'] * 2,
        learning_rate=config['learning_rate'],
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=100,
        warmup_ratio=0.1,
        fp16=True,  # Mixed precision para eficiencia
        report_to="none",  # Desactivar wandb/tensorboard
        seed=SEED,
        save_total_limit=2,  # Mantener solo 2 checkpoints para ahorrar espacio
    )
    
    # 7. Crear Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    
    # 8. Entrenar
    print("\n[5/6] Iniciando entrenamiento...")
    print(f"   Total de pasos: {len(train_tokenized) // config['batch_size'] * config['epochs']}")
    print("="*70)
    
    train_result = trainer.train()
    
    # 9. Evaluar
    print("\n[6/6] Evaluando en validacion...")
    eval_result = trainer.evaluate()
    
    # 10. Guardar modelo
    best_model_path = output_dir / model_key / "best_model"
    trainer.save_model(str(best_model_path))
    print(f"\nModelo guardado en {best_model_path}")
    
    # Limpiar VRAM
    del model
    del trainer
    torch.cuda.empty_cache()
    
    return {
        'model_key': model_key,
        'train_loss': train_result.training_loss,
        'eval_metrics': eval_result,
        'config': config,
    }

print("Funcion train_model() definida.")

Funcion train_model() definida.


## 6. Funciones para Mostrar Resultados

In [14]:
# Funciones para mostrar y guardar resultados de entrenamiento

def display_training_results(result: dict):
    """Muestra tabla detallada de resultados de entrenamiento.
    
    Args:
        result: Diccionario con resultados del entrenamiento
    """
    print(f"\n{'='*70}")
    print(f"RESULTADOS DETALLADOS: {result['model_key']}")
    print(f"{'='*70}")
    
    metrics = result['eval_metrics']
    
    # Tabla de metricas
    table = pd.DataFrame({
        'Metrica': ['F1 Macro', 'Accuracy', 'Precision (macro)', 'Recall (macro)', 
                    'F1 Clase 1 (cambio)', 'Eval Loss', 'Train Loss'],
        'Valor': [
            f"{metrics['eval_f1_macro']:.4f}",
            f"{metrics['eval_accuracy']:.4f}",
            f"{metrics['eval_precision_macro']:.4f}",
            f"{metrics['eval_recall_macro']:.4f}",
            f"{metrics['eval_f1_class1']:.4f}",
            f"{metrics['eval_loss']:.4f}",
            f"{result['train_loss']:.4f}",
        ]
    })
    print("\n" + table.to_string(index=False))
    
    # Comparacion con referencias
    print(f"\n{'-'*70}")
    print("COMPARACION CON REFERENCIAS:")
    print(f"{'-'*70}")
    bert_cnn_f1 = 0.707
    qwen_icl_f1 = 0.383  # Mejor ICL del notebook 11
    
    print(f"  E3 bert_cnn (frozen):     F1 = {bert_cnn_f1:.3f}")
    print(f"  ICL qwen3:1.7b (5-shot):  F1 = {qwen_icl_f1:.3f}")
    print(f"  {result['model_key']:20s}  F1 = {metrics['eval_f1_macro']:.3f}")
    
    delta_e3 = metrics['eval_f1_macro'] - bert_cnn_f1
    delta_icl = metrics['eval_f1_macro'] - qwen_icl_f1
    
    print(f"\n  Delta vs E3 (bert_cnn):  {delta_e3:+.3f} ({delta_e3/bert_cnn_f1*100:+.1f}%)")
    print(f"  Delta vs ICL:            {delta_icl:+.3f} ({delta_icl/qwen_icl_f1*100:+.1f}%)")
    
    if metrics['eval_f1_macro'] > bert_cnn_f1:
        print(f"\n  SUPERA A E3 bert_cnn")
    
    print(f"{'='*70}\n")


def save_incremental_results(results: dict, path: Path):
    """Guarda resultados incrementales.
    
    Args:
        results: Diccionario con resultados de todos los modelos
        path: Ruta donde guardar el JSON
    """
    to_save = {}
    for model_key, result in results.items():
        to_save[model_key] = {
            'eval_metrics': result['eval_metrics'],
            'train_loss': result['train_loss'],
            'config': result['config'],
        }
    
    with open(path, 'w') as f:
        json.dump(to_save, f, indent=2, default=str)
    
    print(f"  Resultados guardados en {path.name}")

print("Funciones de visualizacion definidas.")

Funciones de visualizacion definidas.


In [15]:
# Inicializo el diccionario donde guardare los resultados de todos los modelos
# IMPORTANTE: Ejecutar esta celda UNA VEZ antes de entrenar modelos
# Si ya has entrenado algunos modelos y quieres continuar, NO ejecutes esta celda

all_results = {}

print("="*70)
print("DICCIONARIO DE RESULTADOS INICIALIZADO")
print("="*70)
print("\nAhora ejecuta las celdas de entrenamiento de cada modelo.")
print("   Puedes ejecutarlas todas seguidas o una a una.")
print("\nIMPORTANTE: Si ya has entrenado algunos modelos y quieres continuar,")
print("   NO ejecutes esta celda (perderias los resultados).")
print("="*70)

DICCIONARIO DE RESULTADOS INICIALIZADO

Ahora ejecuta las celdas de entrenamiento de cada modelo.
   Puedes ejecutarlas todas seguidas o una a una.

IMPORTANTE: Si ya has entrenado algunos modelos y quieres continuar,
   NO ejecutes esta celda (perderias los resultados).


## 7. Entrenamiento por Modelo (celdas separadas)

**Instrucciones:**
- Cada celda entrena un modelo independientemente
- Puedes ejecutar todas seguidas (tarda varias horas) o una a una
- Los resultados se guardan incrementalmente despues de cada modelo
- Los checkpoints se guardan en `checkpoints/finetuning/{model_key}/`

### 7.1 DistilBERT (66M params)

In [14]:
# Entreno DistilBERT con full fine-tuning
# Tecnica: Full fine-tuning (todos los parametros)
# VRAM: aproximadamente 3-4GB
# Tiempo estimado: 30-45 minutos

MODEL_KEY = "distilbert"

if MODEL_KEY in MODELS_TO_TRAIN:
    # Entrenar modelo
    results_distilbert = train_model(MODEL_KEY, train_dataset, val_dataset, CHECKPOINTS_DIR)
    
    # Mostrar resultados detallados
    display_training_results(results_distilbert)
    
    # Guardar en diccionario global
    all_results[MODEL_KEY] = results_distilbert
    
    # Guardar metricas incrementales
    save_incremental_results(all_results, REPORTS_DIR / "12_finetuning_metrics_partial.json")
    
    print(f"{MODEL_KEY} completado\n")
else:
    print(f"Saltando {MODEL_KEY} (no esta en MODELS_TO_TRAIN)")


ENTRENANDO: distilbert
  Modelo base: distilbert-base-uncased
  66M params - Full fine-tuning
  QLoRA: False
  Batch size: 32
  Learning rate: 2e-05
  Epochs: 3

[1/6] Cargando tokenizer...
   Tipo de modelo: Encoder

[2/6] Cargando modelo...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   Parametros totales: 66,955,010
   Parametros entrenables: 66,955,010

[3/6] Tokenizando datasets...


Tokenizando train:   0%|          | 0/156695 [00:00<?, ? examples/s]

Tokenizando val:   0%|          | 0/33298 [00:00<?, ? examples/s]

   Train tokenizado: 156,695 ejemplos
   Val tokenizado: 33,298 ejemplos

[4/6] Configurando entrenamiento...

[5/6] Iniciando entrenamiento...
   Total de pasos: 14688


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Class1,Precision Macro,Recall Macro
1,0.431500,0.433798,0.821731,0.575134,0.251450,0.778785,0.569071
2,0.397000,0.436460,0.821190,0.581666,0.265120,0.763584,0.573155
3,0.352400,0.466949,0.812842,0.606296,0.321133,0.699157,0.590924



[6/6] Evaluando en validacion...



Modelo guardado en /home/eeguskiza/DEUSTO/multi-author-analysis/checkpoints/finetuning/distilbert/best_model

RESULTADOS DETALLADOS: distilbert

            Metrica  Valor
           F1 Macro 0.6063
           Accuracy 0.8128
  Precision (macro) 0.6992
     Recall (macro) 0.5909
F1 Clase 1 (cambio) 0.3211
          Eval Loss 0.4669
         Train Loss 0.4045

----------------------------------------------------------------------
COMPARACION CON REFERENCIAS:
----------------------------------------------------------------------
  E3 bert_cnn (frozen):     F1 = 0.707
  ICL qwen3:1.7b (5-shot):  F1 = 0.383
  distilbert            F1 = 0.606

  Delta vs E3 (bert_cnn):  -0.101 (-14.2%)
  Delta vs ICL:            +0.223 (+58.3%)

  Resultados guardados en 12_finetuning_metrics_partial.json
distilbert completado



### 7.2 Qwen3-0.6B (600M params)

In [16]:
# Entreno Qwen3-0.6B con full fine-tuning
# Tecnica: Full fine-tuning (todos los parametros)
# VRAM: aproximadamente 5-6GB
# Tiempo estimado: 1 hora

MODEL_KEY = "qwen3-0.6b"

if MODEL_KEY in MODELS_TO_TRAIN:
    # Entrenar modelo
    results_qwen06 = train_model(MODEL_KEY, train_dataset, val_dataset, CHECKPOINTS_DIR)
    
    # Mostrar resultados detallados
    display_training_results(results_qwen06)
    
    # Guardar en diccionario global
    all_results[MODEL_KEY] = results_qwen06
    
    # Guardar metricas incrementales
    save_incremental_results(all_results, REPORTS_DIR / "12_finetuning_metrics_partial.json")
    
    print(f"{MODEL_KEY} completado\n")
else:
    print(f"Saltando {MODEL_KEY} (no esta en MODELS_TO_TRAIN)")


ENTRENANDO: qwen3-0.6b
  Modelo base: Qwen/Qwen3-0.6B
  600M params - Full fine-tuning
  QLoRA: False
  Batch size: 16
  Learning rate: 2e-05
  Epochs: 3

[1/6] Cargando tokenizer...
   Tipo de modelo: Causal (Decoder)

[2/6] Cargando modelo...


Some weights of Qwen3ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen3-0.6B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


   Parametros totales: 596,051,968
   Parametros entrenables: 596,051,968

[3/6] Tokenizando datasets...


Tokenizando train:   0%|          | 0/156695 [00:00<?, ? examples/s]

Tokenizando val:   0%|          | 0/33298 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


   Train tokenizado: 156,695 ejemplos
   Val tokenizado: 33,298 ejemplos

[4/6] Configurando entrenamiento...

[5/6] Iniciando entrenamiento...
   Total de pasos: 29379


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

### 7.3 Qwen3-1.7B (1.7B params)

In [19]:
# Entreno Qwen3-1.7B con QLoRA (4-bit)
# Tecnica: QLoRA (4-bit quantization + Low-Rank Adaptation)
# VRAM: aproximadamente 5-6GB
# Tiempo estimado: 2-3 horas

MODEL_KEY = "qwen3-1.7b"

if MODEL_KEY in MODELS_TO_TRAIN:
    # Entrenar modelo
    results_qwen17 = train_model(MODEL_KEY, train_dataset, val_dataset, CHECKPOINTS_DIR)
    
    # Mostrar resultados detallados
    display_training_results(results_qwen17)
    
    # Guardar en diccionario global
    all_results[MODEL_KEY] = results_qwen17
    
    # Guardar metricas incrementales
    save_incremental_results(all_results, REPORTS_DIR / "12_finetuning_metrics_partial.json")
    
    print(f"{MODEL_KEY} completado\n")
else:
    print(f"Saltando {MODEL_KEY} (no esta en MODELS_TO_TRAIN)")


ENTRENANDO: qwen3-1.7b
  Modelo base: Qwen/Qwen3-1.7B
  1.7B params - QLoRA (4-bit)
  QLoRA: True
  Batch size: 8
  Learning rate: 0.0001
  Epochs: 3

[1/6] Cargando tokenizer...
   Tipo de modelo: Causal (Decoder)

[2/6] Cargando modelo...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/622M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

Tokenizando val:   0%|          | 0/33298 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


   Train tokenizado: 156,695 ejemplos
   Val tokenizado: 33,298 ejemplos

[4/6] Configurando entrenamiento...

[5/6] Iniciando entrenamiento...
   Total de pasos: 58758


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## 8. Evaluacion por Nivel de Dificultad

Ahora que tengo los modelos entrenados, los evaluo por separado en cada nivel (easy, medium, hard) para ver como varia el rendimiento segun la complejidad del documento.

In [ ]:
# Funcion para evaluar un modelo entrenado por nivel de dificultad
# Similar al notebook 11 pero adaptado para modelos fine-tuned

def evaluate_model_by_level(model_path: Path, tokenizer, val_dataset: Dataset, model_key: str) -> dict:
    """Evalua un modelo fine-tuned por nivel de dificultad.
    
    Args:
        model_path: Ruta al checkpoint del modelo
        tokenizer: Tokenizer del modelo
        val_dataset: Dataset de validacion con columnas level, label
        model_key: Clave del modelo (para determinar si es causal)
    
    Returns:
        dict: {'easy': {...}, 'medium': {...}, 'hard': {...}}
    """
    # Cargar modelo entrenado
    print(f"\nCargando modelo desde {model_path}...")
    
    config = CONFIGS[model_key]
    use_qlora = config.get("use_qlora", False)
    
    if use_qlora:
        # Para QLoRA, cargar el modelo base cuantizado y anadir adaptadores
        from peft import PeftModel
        
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        
        base_model = AutoModelForSequenceClassification.from_pretrained(
            config["model_name"],
            num_labels=2,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
        )
        model = PeftModel.from_pretrained(base_model, model_path)
    else:
        # Full fine-tuning: cargar directamente
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path,
            trust_remote_code=True,
        )
        model.to("cuda")
    
    model.eval()
    
    # Tokenizar dataset completo
    is_causal = "qwen" in config["model_name"].lower()
    tokenize_fn = get_tokenize_function_causal(tokenizer) if is_causal else get_tokenize_function(tokenizer)
    
    val_tokenized = val_dataset.map(
        tokenize_fn,
        batched=True,
        remove_columns=['text_a', 'text_b'],
        desc="Tokenizando validacion"
    )
    
    # Configurar Trainer solo para evaluacion
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    
    trainer = Trainer(
        model=model,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    
    # Obtener predicciones en todo el dataset
    print(f"Evaluando en {len(val_tokenized):,} ejemplos...")
    predictions_output = trainer.predict(val_tokenized)
    predictions = np.argmax(predictions_output.predictions, axis=1)
    
    # Crear DataFrame con nivel, label y prediccion
    eval_df = pd.DataFrame({
        'level': val_dataset['level'],
        'label': val_dataset['label'],
        'prediction': predictions
    })
    
    # Calcular metricas por nivel
    results_by_level = {}
    
    for level in ['easy', 'medium', 'hard']:
        level_df = eval_df[eval_df['level'] == level]
        
        if len(level_df) == 0:
            continue
        
        y_true = level_df['label'].values
        y_pred = level_df['prediction'].values
        
        # Calcular metricas para este nivel
        results_by_level[level] = {
            'accuracy': accuracy_score(y_true, y_pred),
            'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
            'f1_class1': f1_score(y_true, y_pred, average='binary', zero_division=0),
            'precision_macro': precision_score(y_true, y_pred, average='macro', zero_division=0),
            'recall_macro': recall_score(y_true, y_pred, average='macro', zero_division=0),
            'n_examples': len(level_df),
            'n_class_0': (y_true == 0).sum(),
            'n_class_1': (y_true == 1).sum(),
        }
    
    # Limpiar memoria
    del model
    del trainer
    torch.cuda.empty_cache()
    
    return results_by_level

print("Funcion evaluate_model_by_level() definida.")

In [ ]:
# Ejecuto la evaluacion por nivel para todos los modelos entrenados
# Esto puede tardar 10-20 minutos dependiendo de cuantos modelos se entrenaron

if all_results:
    print("="*70)
    print("EVALUANDO MODELOS POR NIVEL DE DIFICULTAD")
    print("="*70)
    
    results_by_level = {}
    
    for model_key in all_results.keys():
        print(f"\nEvaluando {model_key} por nivel...")
        
        # Ruta al mejor checkpoint
        model_path = CHECKPOINTS_DIR / model_key / "best_model"
        
        if not model_path.exists():
            print(f"  Checkpoint no encontrado en {model_path}")
            continue
        
        # Cargar tokenizer
        config = CONFIGS[model_key]
        tokenizer = AutoTokenizer.from_pretrained(config["model_name"], trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        
        # Evaluar por nivel
        try:
            results_by_level[model_key] = evaluate_model_by_level(
                model_path, tokenizer, val_dataset, model_key
            )
            print(f"  {model_key} completado")
        except Exception as e:
            print(f"  Error evaluando {model_key}: {e}")
            continue
    
    print(f"\nEvaluacion por nivel completada para {len(results_by_level)} modelos.")
    print("="*70)
    
else:
    print("No hay resultados todavia. Ejecuta las celdas de entrenamiento primero.")

## 9. Resumen de Resultados: Agregados y por Nivel

In [ ]:
# Creo tabla resumen con metricas agregadas de todos los modelos

if all_results:
    summary_data = []
    for model_key, result in all_results.items():
        m = result['eval_metrics']
        summary_data.append({
            'Modelo': model_key,
            'Tecnica': 'QLoRA' if result['config'].get('use_qlora') else 'Full FT',
            'F1 Macro': m['eval_f1_macro'],
            'Accuracy': m['eval_accuracy'],
            'F1 Clase 1': m['eval_f1_class1'],
            'Precision': m['eval_precision_macro'],
            'Recall': m['eval_recall_macro'],
            'Epochs': result['config']['epochs'],
            'LR': result['config']['learning_rate'],
        })
    
    summary_df = pd.DataFrame(summary_data).sort_values('F1 Macro', ascending=False)
    
    print("="*100)
    print("RESUMEN AGREGADO: Fine-tuning - Ordenado por F1 Macro")
    print("="*100)
    print(summary_df.to_string(index=False))
    print("="*100)
    
    # Mostrar el mejor modelo
    best = summary_df.iloc[0]
    print(f"\nMejor modelo: {best['Modelo']} con F1 Macro = {best['F1 Macro']:.4f}")
    print(f"   Tecnica: {best['Tecnica']}")
    print(f"   F1 Clase 1 (cambio): {best['F1 Clase 1']:.4f}")
    
    # Comparacion con E3
    bert_cnn_f1 = 0.707
    if best['F1 Macro'] > bert_cnn_f1:
        improvement = best['F1 Macro'] - bert_cnn_f1
        print(f"\n   SUPERA a E3 bert_cnn por {improvement:+.3f} ({improvement/bert_cnn_f1*100:+.1f}%)")
    else:
        gap = bert_cnn_f1 - best['F1 Macro']
        print(f"\n   No supera a E3 bert_cnn (gap: {gap:.3f})")
    
else:
    print("No hay resultados todavia. Ejecuta las celdas de entrenamiento primero.")

In [ ]:
# Creo tabla resumen con metricas desglosadas por nivel de dificultad

if 'results_by_level' in locals() and results_by_level:
    summary_level_data = []
    
    for model_key, levels_data in results_by_level.items():
        for level, metrics in levels_data.items():
            summary_level_data.append({
                'Modelo': model_key,
                'Nivel': level.capitalize(),
                'N': metrics['n_examples'],
                'F1 Macro': metrics['f1_macro'],
                'Accuracy': metrics['accuracy'],
                'F1 Clase 1': metrics['f1_class1'],
                'Precision': metrics['precision_macro'],
                'Recall': metrics['recall_macro'],
            })
    
    summary_level_df = pd.DataFrame(summary_level_data)
    summary_level_df = summary_level_df.sort_values(['Nivel', 'F1 Macro'], ascending=[True, False])
    
    print("\n" + "="*100)
    print("RESUMEN POR NIVEL DE DIFICULTAD")
    print("="*100)
    print(summary_level_df.to_string(index=False))
    print("="*100)
    
    # Analisis: Mejor modelo por nivel
    print("\nMejor modelo por nivel:")
    for level in ['Easy', 'Medium', 'Hard']:
        level_df = summary_level_df[summary_level_df['Nivel'] == level]
        best_level = level_df.iloc[0]
        print(f"  {level:6s}: {best_level['Modelo']:15s} (F1 = {best_level['F1 Macro']:.4f})")
    
else:
    print("No hay resultados por nivel todavia. Ejecuta la evaluacion por nivel primero.")

## 10. Visualizaciones

In [ ]:
# Grafico de F1 Macro por nivel (3 subplots)
# Comparo como varia el rendimiento de cada modelo segun la dificultad

if 'results_by_level' in locals() and results_by_level:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    levels = ['easy', 'medium', 'hard']
    model_keys = sorted(results_by_level.keys())
    x = np.arange(len(model_keys))
    
    for idx, level in enumerate(levels):
        ax = axes[idx]
        
        # Extraigo F1 para este nivel
        f1_values = []
        for model_key in model_keys:
            f1 = results_by_level[model_key].get(level, {}).get('f1_macro', 0)
            f1_values.append(f1)
        
        # Creo barras con colores segun el modelo
        colors = ['#4ECDC4' if 'distilbert' in m else '#FF6B6B' if '0.6b' in m else '#95E1D3' if '1.7b' in m else '#F38181' for m in model_keys]
        bars = ax.bar(x, f1_values, color=colors, alpha=0.8)
        
        ax.set_ylabel('F1 Macro', fontsize=11)
        ax.set_title(f'Nivel: {level.upper()}', fontsize=12, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(model_keys, rotation=45, ha='right')
        ax.grid(axis='y', alpha=0.3)
        
        # Valores en barras
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                       f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / '12_f1_by_level.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Grafico guardado en {REPORTS_DIR / '12_f1_by_level.png'}")
else:
    print("No hay resultados por nivel para graficar.")

In [ ]:
# Grafico de comparacion: F1 Macro agregado vs por nivel
# Muestra como varia el rendimiento de cada modelo en los diferentes niveles

if all_results and 'results_by_level' in locals() and results_by_level:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    model_keys = sorted(all_results.keys())
    x = np.arange(len(model_keys))
    width = 0.2
    
    # Extraigo F1 agregado y por nivel para cada modelo
    f1_aggregated = [all_results[m]['eval_metrics']['eval_f1_macro'] for m in model_keys]
    f1_easy = [results_by_level.get(m, {}).get('easy', {}).get('f1_macro', 0) for m in model_keys]
    f1_medium = [results_by_level.get(m, {}).get('medium', {}).get('f1_macro', 0) for m in model_keys]
    f1_hard = [results_by_level.get(m, {}).get('hard', {}).get('f1_macro', 0) for m in model_keys]
    
    # Creo barras agrupadas
    ax.bar(x - width*1.5, f1_aggregated, width, label='Agregado', color='#95E1D3', alpha=0.9)
    ax.bar(x - width*0.5, f1_easy, width, label='Easy', color='#4ECDC4', alpha=0.9)
    ax.bar(x + width*0.5, f1_medium, width, label='Medium', color='#F38181', alpha=0.9)
    ax.bar(x + width*1.5, f1_hard, width, label='Hard', color='#FF6B6B', alpha=0.9)
    
    ax.set_ylabel('F1 Macro', fontsize=12)
    ax.set_title('Comparacion F1 Macro: Agregado vs Por Nivel', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(model_keys, rotation=45, ha='right')
    ax.legend(loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / '12_f1_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Grafico guardado en {REPORTS_DIR / '12_f1_comparison.png'}")
else:
    print("No hay resultados completos para graficar.")

In [ ]:
# Heatmap de F1 por nivel
# Visualizacion compacta de todos los modelos y niveles

if 'results_by_level' in locals() and results_by_level:
    f1_by_level_data = []
    model_labels = []
    
    for model_key in sorted(results_by_level.keys()):
        model_labels.append(model_key)
        levels_data = results_by_level[model_key]
        f1_by_level_data.append([
            levels_data.get('easy', {}).get('f1_macro', 0),
            levels_data.get('medium', {}).get('f1_macro', 0),
            levels_data.get('hard', {}).get('f1_macro', 0),
        ])
    
    f1_level_df = pd.DataFrame(
        f1_by_level_data,
        index=model_labels,
        columns=['Easy', 'Medium', 'Hard']
    )
    
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(f1_level_df.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=max(0.5, f1_level_df.values.max()))
    
    ax.set_xticks(np.arange(len(f1_level_df.columns)))
    ax.set_yticks(np.arange(len(f1_level_df.index)))
    ax.set_xticklabels(f1_level_df.columns)
    ax.set_yticklabels(f1_level_df.index)
    
    plt.setp(ax.get_xticklabels(), rotation=0, ha="center")
    
    # Anadir valores en cada celda
    for i in range(len(f1_level_df.index)):
        for j in range(len(f1_level_df.columns)):
            text = ax.text(j, i, f'{f1_level_df.values[i, j]:.3f}',
                          ha="center", va="center", color="black", fontsize=10)
    
    ax.set_title('Heatmap F1 Macro POR NIVEL', fontweight='bold', fontsize=12)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(REPORTS_DIR / '12_f1_heatmap_by_level.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Heatmap guardado en {REPORTS_DIR / '12_f1_heatmap_by_level.png'}")
else:
    print("No hay resultados por nivel para crear heatmap.")

## 11. Comparacion con E3 e ICL

In [ ]:
# Comparacion completa: Fine-tuning vs E3 vs ICL
# Esta es una comparacion preliminar - la comparacion completa va en el notebook 13

if all_results:
    print("\n" + "="*70)
    print("COMPARACION: Fine-tuning vs E3 vs ICL")
    print("="*70)
    print(f"\n{'Modelo':<30} {'F1 Macro':>10} {'Paradigma':>20}")
    print("-" * 70)
    
    # Modelos de referencia
    print(f"{'bert_cnn (E3 frozen)':<30} {'0.707':>10} {'Frozen+CNN':>20}")
    print(f"{'bert_lstm (E3 frozen)':<30} {'0.685':>10} {'Frozen+LSTM':>20}")
    print(f"{'qwen3:1.7b ICL (5-shot)':<30} {'0.383':>10} {'Zero-shot ICL':>20}")
    
    print("-" * 70)
    
    # Modelos fine-tuned ordenados por F1
    for model_key in sorted(all_results.keys(), key=lambda k: all_results[k]['eval_metrics']['eval_f1_macro'], reverse=True):
        result = all_results[model_key]
        f1 = result['eval_metrics']['eval_f1_macro']
        tecnica = 'QLoRA' if result['config'].get('use_qlora') else 'Full FT'
        print(f"{model_key + ' (fine-tuned)':<30} {f1:>10.3f} {tecnica:>20}")
    
    print("="*70)
    
    print("\nNOTAS:")
    print("  - E3: Frozen BERT + clasificador entrenado (33,858 val examples)")
    print("  - ICL: Zero-shot con 5 ejemplos (500 val examples)")
    print(f"  - Fine-tuning: End-to-end ({len(val_dataset):,} val examples)")
    print("  - Comparacion completa con analisis estadistico en notebook 13")
    
else:
    print("No hay resultados todavia. Ejecuta las celdas de entrenamiento primero.")

## 12. Guardar Resultados Finales

In [ ]:
# Guardo las metricas finales: agregadas + por nivel
# Estos resultados se usaran en el notebook 13 para comparacion completa

if all_results:
    # Construir diccionario con metricas completas
    final_metrics = {}
    for model_key, result in all_results.items():
        final_metrics[model_key] = {
            'eval_metrics_aggregated': result['eval_metrics'],
            'eval_metrics_by_level': results_by_level.get(model_key, {}),
            'config': result['config'],
            'train_loss': result['train_loss'],
        }

    # Anadir metadatos del experimento
    final_metrics['_metadata'] = {
        'n_train': len(train_dataset),
        'n_val': len(val_dataset),
        'seed': SEED,
        'max_length': MAX_LENGTH,
        'val_distribution_by_level': {
            level: len([x for x in val_dataset if x['level'] == level])
            for level in ['easy', 'medium', 'hard']
        },
        'note': 'Fine-tuning experiments with full training and QLoRA. Includes aggregated and level-based metrics.'
    }

    # Guardar en JSON
    with open(REPORTS_DIR / "12_finetuning_metrics_final.json", "w") as f:
        json.dump(final_metrics, f, indent=2, default=str)

    print("="*70)
    print("RESULTADOS FINALES GUARDADOS")
    print("="*70)
    print(f"\nMetricas guardadas en {REPORTS_DIR / '12_finetuning_metrics_final.json'}")
    print(f"\nCheckpoints guardados en {CHECKPOINTS_DIR}")
    
    # Listar checkpoints guardados
    print("\nCheckpoints disponibles:")
    for model_key in all_results.keys():
        checkpoint_path = CHECKPOINTS_DIR / model_key / "best_model"
        if checkpoint_path.exists():
            # Calcular tamano del checkpoint
            size = sum(f.stat().st_size for f in checkpoint_path.rglob('*') if f.is_file())
            print(f"  - {model_key}: {size / 1e6:.1f} MB")
    
    print("\nProcede al notebook 13 para comparacion completa y analisis estadistico.")
    print("="*70)
    
else:
    print("No hay resultados para guardar. Ejecuta las celdas de entrenamiento primero.")